<a href="https://colab.research.google.com/github/Vickayo04/Machine_Learning-Projects/blob/main/Solve_Business_Problems_with_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Solve Business Problems with AI

## Objective
Develop a proof-of-concept application to intelligently process email order requests and customer inquiries for a fashion store. The system should accurately categorize emails as either product inquiries or order requests and generate appropriate responses using the product catalog information and current stock status.

You are encouraged to use AI assistants (like ChatGPT or Claude) and any IDE of your choice to develop your solution. Many modern IDEs (such as PyCharm, or Cursor) can work with Jupiter files directly.

## Task Description

### Inputs

Google Spreadsheet **[Document](https://docs.google.com/spreadsheets/d/14fKHsblfqZfWj3iAaM2oA51TlYfQlFT4WKo52fVaQ9U)** containing:

- **Products**: List of products with fields including product ID, name, category, stock amount, detailed description, and season.

- **Emails**: Sequential list of emails with fields such as email ID, subject, and body.

### Instructions

- Implement all requirements using advanced Large Language Models (LLMs) to handle complex tasks, process extensive data, and generate accurate outputs effectively.
- Use Retrieval-Augmented Generation (RAG) and vector store techniques where applicable to retrieve relevant information and generate responses.
- You are provided with a temporary OpenAI API key granting access to GPT-4o, which has a token quota. Use it wisely or use your own key if preferred.
- Address the requirements in the order listed. Review them in advance to develop a general implementation plan before starting.
- Your deliverables should include:
   - Code developed within this notebook.
   - A single spreadsheet containing results, organized across separate sheets.
   - Comments detailing your thought process.
- You may use additional libraries (e.g., langchain) to streamline the solution. Use libraries appropriately to align with best practices for AI and LLM tools.
- Use the most suitable AI techniques for each task. Note that solving tasks with traditional programming methods will not earn points, as this assessment evaluates your knowledge of LLM tools and best practices.

### Requirements

#### 1. Classify emails
    
Classify each email as either a _**"product inquiry"**_ or an _**"order request"**_. Ensure that the classification accurately reflects the intent of the email.

**Output**: Populate the **email-classification** sheet with columns: email ID, category.

#### 2. Process order requests
1.   Process orders
  - For each order request, verify product availability in stock.
  - If the order can be fulfilled, create a new order line with the status “created”.
  - If the order cannot be fulfilled due to insufficient stock, create a line with the status “out of stock” and include the requested quantity.
  - Update stock levels after processing each order.
  - Record each product request from the email.
  - **Output**: Populate the **order-status** sheet with columns: email ID, product ID, quantity, status (**_"created"_**, **_"out of stock"_**).

2.   Generate responses
  - Create response emails based on the order processing results:
      - If the order is fully processed, inform the customer and provide product details.
      - If the order cannot be fulfilled or is only partially fulfilled, explain the situation, specify the out-of-stock items, and suggest alternatives or options (e.g., waiting for restock).
  - Ensure the email tone is professional and production-ready.
  - **Output**: Populate the **order-response** sheet with columns: email ID, response.

#### 3. Handle product inquiry

Customers may ask general open questions.
  - Respond to product inquiries using relevant information from the product catalog.
  - Ensure your solution scales to handle a full catalog of over 100,000 products without exceeding token limits. Avoid including the entire catalog in the prompt.
  - **Output**: Populate the **inquiry-response** sheet with columns: email ID, response.

## Evaluation Criteria
- **Advanced AI Techniques**: The system should use Retrieval-Augmented Generation (RAG) and vector store techniques to retrieve relevant information from data sources and use it to respond to customer inquiries.
- **Tone Adaptation**: The AI should adapt its tone appropriately based on the context of the customer's inquiry. Responses should be informative and enhance the customer experience.
- **Code Completeness**: All functionalities outlined in the requirements must be fully implemented and operational as described.
- **Code Quality and Clarity**: The code should be well-organized, with clear logic and a structured approach. It should be easy to understand and maintain.
- **Presence of Expected Outputs**: All specified outputs must be correctly generated and saved in the appropriate sheets of the output spreadsheet. Ensure the format of each output matches the requirements—do not add extra columns or sheets.
- **Accuracy of Outputs**: The accuracy of the generated outputs is crucial and will significantly impact the evaluation of your submission.

We look forward to seeing your solution and your approach to solving real-world problems with AI technologies.

# Prerequisites

### Configure OpenAI API Key.

In [1]:
# Install the OpenAI Python package.
%pip install openai httpx==0.27.2

**IMPORTANT: If you are going to use our custom API Key then make sure that you also use custom base URL as in example below. Otherwise it will not work.**

In [2]:
# Code example of OpenAI communication

from openai import OpenAI

client = OpenAI(
    # In order to use provided API key, make sure that models you create point to this custom base URL.
    base_url='https://47v4us7kyypinfb5lcligtc3x40ygqbs.lambda-url.us-east-1.on.aws/v1/',
    # The temporary API key giving access to ChatGPT 4o model. Quotas apply: you have 500'000 input and 500'000 output tokens, use them wisely ;)
    api_key='a0Bfv00000EntbBEAR'
)

completion = client.chat.completions.create(
  model="gpt-4o",
  messages=[
    {"role": "user", "content": "Hello!"}
  ]
)

print(completion.choices[0].message)

ChatCompletionMessage(content='Hello! How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


In [3]:
# Code example of reading input data

import pandas as pd
from IPython.display import display

def read_data_frame(document_id, sheet_name):
    export_link = f"https://docs.google.com/spreadsheets/d/{document_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"
    return  pd.read_csv(export_link)

document_id = '14fKHsblfqZfWj3iAaM2oA51TlYfQlFT4WKo52fVaQ9U'
products_df = read_data_frame(document_id, 'products')
emails_df = read_data_frame(document_id, 'emails')

# Display first 3 rows of each DataFrame
display(products_df.head(3))
display(emails_df.head(3))

,product_id,name,category,description,stock,seasons,price
0,RSG8901,Retro Sunglasses,Accessories,Transport yourself back in time with our retro...,1,"Spring, Summer",26.99
1,SWL2345,Sleek Wallet,Accessories,Keep your essentials organized and secure with...,5,All seasons,30.00
2,VSC6789,Versatile Scarf,Accessories,Add a touch of versatility to your wardrobe wi...,6,"Spring, Fall",23.00


,email_id,subject,message
0,E001,Leather Wallets,"Hi there, I want to order all the remaining LT..."
1,E002,Buy Vibrant Tote with noise,"Good morning, I'm looking to buy the VBT2345 V..."
2,E003,Need your help,"Hello, I need a new bag to carry my laptop and..."


In [4]:
# Code example of generating output document

# Creates a new shared Google Worksheet every invocation with the proper structure
# Note: This code should be executed from the google colab once you are ready, it will not work locally
from google.colab import auth
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe

# Authentication step to create a google client
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Create a new spreadsheet
output_document = gc.create('Solving Business Problems with AI - Output')

# Create 'email-classification' sheet
email_classification_sheet = output_document.add_worksheet(title="email-classification", rows=50, cols=2)
email_classification_sheet.update([['email ID', 'category']], 'A1:B1')

# Create 'order-status' sheet
order_status_sheet = output_document.add_worksheet(title="order-status", rows=50, cols=4)
order_status_sheet.update([['email ID', 'product ID', 'quantity', 'status']], 'A1:D1')

# Create 'order-response' sheet
order_response_sheet = output_document.add_worksheet(title="order-response", rows=50, cols=2)
order_response_sheet.update([['email ID', 'response']], 'A1:B1')

# Create 'inquiry-response' sheet
inquiry_response_sheet = output_document.add_worksheet(title="inquiry-response", rows=50, cols=2)
inquiry_response_sheet.update([['email ID', 'response']], 'A1:B1')

# Share the spreadsheet publicly
output_document.share('', perm_type='anyone', role='reader')

# This is the solution output link, paste it into the submission form
print(f"Shareable link: https://docs.google.com/spreadsheets/d/{output_document.id}")

Shareable link: https://docs.google.com/spreadsheets/d/1cFL_g89nUAX1_ltuWNt6Otrr07-EBKZG-93LmwuvoLs


# Task 1. Classify emails

In [5]:
def classify_email(subject, body):
    prompt = f"""
    Classify the following customer email into exactly one of two categories:
    1. 'product inquiry' (questions about items, recommendations, fit, details)
    2. 'order request' (requests to purchase items, buy specific quantities, place an order)

    Subject: {subject}
    Body: {body}

    Respond with ONLY the category name in lowercase (\"product inquiry\" or \"order request\").
    """
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()

# Apply classification to emails_df using correct column names: 'email_id', 'subject', 'message'
classification_results = []
for idx, row in emails_df.iterrows():
    # Using 'message' instead of 'body'
    category = classify_email(row['subject'], row['message'])
    classification_results.append({
        'email ID': row['email_id'], # Using 'email_id' from source, mapping to required 'email ID'
        'category': category
    })

email_classification_df = pd.DataFrame(classification_results)

# Task 2. Process order requests

In [6]:
import json

# Filter order request emails
order_email_ids = email_classification_df[
    email_classification_df['category'] == 'order request'
]['email ID'].tolist()

# Fix: Source column is 'email_id'
order_emails = emails_df[emails_df['email_id'].isin(order_email_ids)]

# Active inventory working copy
inventory_df = products_df.copy()

order_status_rows = []
order_responses = []

for idx, email in order_emails.iterrows():
    # 1. Parse products & quantities requested using LLM JSON output
    parse_prompt = f"""
    Extract the requested product names/IDs and requested quantities from this email order.
    Here is our product catalog for context:
    {inventory_df[['product_id', 'name']].to_dict(orient='records')}

    Email Subject: {email['subject']}
    Email Body: {email['message']}

    Return a JSON array of objects with keys \"product ID\" and \"quantity\".
    Output ONLY valid JSON without markdown formatting.
    """

    res = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": parse_prompt}],
        temperature=0
    )

    raw_json = res.choices[0].message.content.strip().replace('```json', '').replace('```', '')
    parsed_items = json.loads(raw_json)

    item_fulfillment_summary = []

    # 2. Check stock & update inventory
    for item in parsed_items:
        p_id = item['product ID']
        qty = int(item['quantity'])

        # Fix: Source column is 'product_id'
        prod_row = inventory_df[inventory_df['product_id'] == p_id]
        if not prod_row.empty:
            # Fix: Source column name is 'stock', not 'stock amount'
            current_stock = prod_row.iloc[0]['stock']
            p_name = prod_row.iloc[0]['name']

            if current_stock >= qty:
                status = 'created'
                # Deduct stock
                inventory_df.loc[inventory_df['product_id'] == p_id, 'stock'] -= qty
            else:
                status = 'out of stock'

            order_status_rows.append({
                'email ID': email['email_id'],
                'product ID': p_id,
                'quantity': qty,
                'status': status
            })
            item_fulfillment_summary.append(f"Item: {p_name} (ID: {p_id}), Qty: {qty}, Status: {status}")

    # 3. Generate Order Response Email
    response_prompt = f"""
    Draft a professional customer service response email.
    Customer request: {email['message']}
    Order outcome summary:
    {chr(10).join(item_fulfillment_summary)}

    If any item was 'out of stock', apologize, explain the delay, and suggest alternatives or waiting for a restock.
    Keep the tone polite, helpful, and ready for production use.
    """

    resp_res = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": response_prompt}],
        temperature=0.7
    )

    order_responses.append({
        'email ID': email['email_id'],
        'response': resp_res.choices[0].message.content.strip()
    })

order_status_df = pd.DataFrame(order_status_rows)
order_response_df = pd.DataFrame(order_responses)

# Task 3. Handle product inquiry

In [7]:
# Install RAG dependencies
%pip install langchain-community faiss-cpu sentence-transformers langchain-huggingface

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import pandas as pd

# Create product descriptions text for embedding using correct source column names
# Source columns: 'product_id', 'name', 'category', 'seasons', 'description', 'stock'
inventory_df['combined_text'] = inventory_df.apply(
    lambda r: f"Product ID: {r['product_id']}, Name: {r['name']}, Category: {r['category']}, Season: {r['seasons']}, Description: {r['description']}, Stock: {r['stock']}",
    axis=1
)

# Build Vector Store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_texts(inventory_df['combined_text'].tolist(), embeddings)

inquiry_email_ids = email_classification_df[
    email_classification_df['category'] == 'product inquiry'
]['email ID'].tolist()

# Source column is 'email_id'
inquiry_emails = emails_df[emails_df['email_id'].isin(inquiry_email_ids)]
inquiry_responses = []

for idx, email in inquiry_emails.iterrows():
    # Retrieve relevant products using 'message' instead of 'body'
    query = f"{email['subject']} {email['message']}"
    docs = vectorstore.similarity_search(query, k=3)
    retrieved_context = "\n\n".join([d.page_content for d in docs])

    inquiry_prompt = f"""
    Answer the following customer inquiry using ONLY the relevant catalog information provided below.
    Adapt your tone appropriately to be polite, informative, and enhance the customer experience.

    Customer Email Subject: {email['subject']}
    Customer Email Body: {email['message']}

    Relevant Catalog Products Context:
    {retrieved_context}
    """

    res = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": inquiry_prompt}],
        temperature=0.7
    )

    inquiry_responses.append({
        'email ID': email['email_id'],
        'response': res.choices[0].message.content.strip()
    })

inquiry_response_df = pd.DataFrame(inquiry_responses)

/tmp/ipykernel_5753/4048168228.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
from google.colab import auth
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

output_document = gc.create('Solving Business Problems with AI - Output')

# 1. Sheet: email-classification
s1 = output_document.add_worksheet(title="email-classification", rows=100, cols=2)
set_with_dataframe(s1, email_classification_df)

# 2. Sheet: order-status
s2 = output_document.add_worksheet(title="order-status", rows=100, cols=4)
set_with_dataframe(s2, order_status_df)

# 3. Sheet: order-response
s3 = output_document.add_worksheet(title="order-response", rows=100, cols=2)
set_with_dataframe(s3, order_response_df)

# 4. Sheet: inquiry-response
s4 = output_document.add_worksheet(title="inquiry-response", rows=100, cols=2)
set_with_dataframe(s4, inquiry_response_df)

# Delete default Sheet1 created by default
default_sheet = output_document.worksheet("Sheet1")
output_document.del_worksheet(default_sheet)

# Set Share permissions
output_document.share('', perm_type='anyone', role='reader')

print(f"Shareable Output Link: https://docs.google.com/spreadsheets/d/{output_document.id}")

Shareable Output Link: https://docs.google.com/spreadsheets/d/1Akh-fWYmTz4NjqZ3s6ntSCVz5L_njLyNNM5Kk9WUidA
